In [23]:
import math
from typing import List, Tuple, Optional

#Formato para el tablero
USE_UNICODE = True
BORDES_UNICODE = {
    "TL": "┌", "TR": "┐", "BL": "└", "BR": "┘",
    "H": "─", "V": "│", "TJ": "┬", "BJ": "┴",
    "LJ": "├", "RJ": "┤", "CJ": "┼"
}
BORDES_ASCII = {
    "TL": "+", "TR": "+", "BL": "+", "BR": "+",
    "H": "-", "V": "|", "TJ": "+", "BJ": "+",
    "LJ": "+", "RJ": "+", "CJ": "+"
}
BORDES = BORDES_UNICODE if USE_UNICODE else BORDES_ASCII


MOSTRAR_ARBOL = False
NIVELES_ARBOL = 2
USAR_HEURISTICA_TIE = True

estados_visitados = 0
profundidad_maxima = 0
historial_partidas = []

def imprimir_tablero(tablero: List[str]) -> None:
    print(f"{BORDES['TL']}{BORDES['H']*3}{BORDES['TJ']}{BORDES['H']*3}{BORDES['TJ']}{BORDES['H']*3}{BORDES['TR']}")
    for i in range(3):
        fila = f"{BORDES['V']} {tablero[i*3]} {BORDES['V']} {tablero[i*3+1]} {BORDES['V']} {tablero[i*3+2]} {BORDES['V']}"
        print(fila)
        if i < 2:
            print(f"{BORDES['LJ']}{BORDES['H']*3}{BORDES['CJ']}{BORDES['H']*3}{BORDES['CJ']}{BORDES['H']*3}{BORDES['RJ']}")
    print(f"{BORDES['BL']}{BORDES['H']*3}{BORDES['BJ']}{BORDES['H']*3}{BORDES['BJ']}{BORDES['H']*3}{BORDES['BR']}")

def verificar_ganador(tablero: List[str]) -> Optional[str]:
    combinaciones = [
        (0,1,2),(3,4,5),(6,7,8),
        (0,3,6),(1,4,7),(2,5,8),
        (0,4,8),(2,4,6)
    ]
    for a,b,c in combinaciones:
        if tablero[a] == tablero[b] == tablero[c] and tablero[a] != " ":
            return tablero[a]
    if " " not in tablero:
        return "Empate"
    return None

# Heuristica
def valor_heuristico(tablero: List[str]) -> float:
    # +2 por linea con 2 en O y 1 vacio
    #-2 por 2 en X y 1 vacío (bloqueo)
    # +0.5 por cada esquina que esta ocupada por O
    # +1 por centro ocupado por O
    lineas = [
        (0,1,2),(3,4,5),(6,7,8),
        (0,3,6),(1,4,7),(2,5,8),
        (0,4,8),(2,4,6)
    ]
    h = 0.0
    for a,b,c in lineas:
        linea = [tablero[a], tablero[b], tablero[c]]
        if linea.count("O") == 2 and linea.count(" ") == 1:
            h += 2
        if linea.count("X") == 2 and linea.count(" ") == 1:
            h -= 2
# Centro
    if tablero[4] == "O":
        h += 1
# Esquinas
    for i in [0,2,6,8]:
        if tablero[i] == "O":
            h += 0.5
    return h



# + (1: contadores) + (2: ruta) + (historial de estados)
def minimax(tablero: List[str],
            profundidad: int = 0,
            es_maximizando: bool = True,
            historial_estado: Optional[List[dict]] = None) -> Tuple[int, List[Tuple[str, int]]]:
    global estados_visitados, profundidad_maxima
    estados_visitados += 1
    if profundidad > profundidad_maxima:
        profundidad_maxima = profundidad

    ganador = verificar_ganador(tablero)
    if ganador == "O":
        puntaje = 10 - profundidad
        if historial_estado is not None:
            historial_estado.append({"tablero": tablero.copy(), "puntaje": puntaje, "profundidad": profundidad})
        return puntaje, []
    if ganador == "X":
        puntaje = -10 + profundidad
        if historial_estado is not None:
            historial_estado.append({"tablero": tablero.copy(), "puntaje": puntaje, "profundidad": profundidad})
        return puntaje, []
    if ganador == "Empate":
        puntaje = 0
        if historial_estado is not None:
            historial_estado.append({"tablero": tablero.copy(), "puntaje": puntaje, "profundidad": profundidad})
        return 0, []

    if historial_estado is not None:
        historial_estado.append({"tablero": tablero.copy(), "puntaje": None, "profundidad": profundidad})

    if es_maximizando:
        mejor_puntaje = -math.inf
        mejor_ruta: List[Tuple[str,int]] = []
        mejores_candidatos: List[Tuple[int, int, List[Tuple[str,int]], float]] = []
        for i in range(9):
            if tablero[i] == " ":
                tablero[i] = "O"
                p, ruta_hija = minimax(tablero, profundidad + 1, False, historial_estado)
                tablero[i] = " "
# Guardar
                heur = valor_heuristico(tablero) if USAR_HEURISTICA_TIE else 0.0
                mejores_candidatos.append((p, i, ruta_hija, heur))
                if p > mejor_puntaje:
                    mejor_puntaje = p
# Desempate mediante la heuristica si es que hay puntajes iguales
        candidatos_mejor = [c for c in mejores_candidatos if c[0] == mejor_puntaje]
        if USAR_HEURISTICA_TIE and len(candidatos_mejor) > 1:
# Elige el de mayor heuristica
            p, pos, ruta_hija, _ = max(candidatos_mejor, key=lambda x: x[3])
        else:
            p, pos, ruta_hija, _ = candidatos_mejor[0]
        mejor_ruta = [("O", pos)] + ruta_hija
        return mejor_puntaje, mejor_ruta
    else:
        mejor_puntaje = math.inf
        mejor_ruta = []
        mejores_candidatos = []
        for i in range(9):
            if tablero[i] == " ":
                tablero[i] = "X"
                p, ruta_hija = minimax(tablero, profundidad + 1, True, historial_estado)
                tablero[i] = " "
                heur = valor_heuristico(tablero) if USAR_HEURISTICA_TIE else 0.0
                mejores_candidatos.append((p, i, ruta_hija, heur))
                if p < mejor_puntaje:
                    mejor_puntaje = p

        candidatos_mejor = [c for c in mejores_candidatos if c[0] == mejor_puntaje]
        p, pos, ruta_hija, _ = candidatos_mejor[0]
        mejor_ruta = [("X", pos)] + ruta_hija
        return mejor_puntaje, mejor_ruta

def mejor_movimiento(tablero: List[str], historial_estado: List[dict]) -> Tuple[int, List[Tuple[str,int]]]:
    mejor_puntaje = -math.inf
    mejor_ruta: List[Tuple[str,int]] = []
    candidatos: List[Tuple[int, int, List[Tuple[str,int]], float]] = []
    for i in range(9):
        if tablero[i] == " ":
            tablero[i] = "O"
            p, ruta = minimax(tablero, 0, False, historial_estado)
            tablero[i] = " "
            heur = valor_heuristico(tablero) if USAR_HEURISTICA_TIE else 0.0
            candidatos.append((p, i, ruta, heur))
            if p > mejor_puntaje:
                mejor_puntaje = p
    mejores = [c for c in candidatos if c[0] == mejor_puntaje]
    if USAR_HEURISTICA_TIE and len(mejores) > 1:
        p, pos, ruta, _ = max(mejores, key=lambda x: x[3])
    else:
        p, pos, ruta, _ = mejores[0]
    return pos, [("O", pos)] + ruta


def jugar():
    global estados_visitados, profundidad_maxima
    estados_visitados = 0
    profundidad_maxima = 0

    tablero = [" "] * 9
    historial_estado = []
    mejor_linea_total: List[Tuple[str,int]] = []

    print("=== TIC TAC TOE ===")
    print("Eres X, la computadora es O")
    imprimir_tablero(tablero)

    if MOSTRAR_ARBOL:
        print("\n[Arbol inicial hasta nivel", NIVELES_ARBOL, "]")
        imprimir_arbol(tablero, 0, True, NIVELES_ARBOL)

    while True:
        mov = -1
        while mov not in range(1,10) or tablero[mov-1] != " ":
            try:
                mov = int(input("Elige tu movimiento 1-9: "))
            except:
                mov = -1
        tablero[mov-1] = "X"
        imprimir_tablero(tablero)
        if verificar_ganador(tablero) is not None:
            break

        print("Turno de Computadora")
        pos, ruta = mejor_movimiento(tablero, historial_estado)
# Acumula la mejor linea
        mejor_linea_total += ruta
        tablero[pos] = "O"
        imprimir_tablero(tablero)
        if verificar_ganador(tablero) is not None:
            break

    res = verificar_ganador(tablero)
    if res == "X": print("Gana el jugador (X)")
    elif res == "O": print("Gana la computadora (O)")
    else: print("Empate")

    historial_partidas.append({
        "tablero_final": tablero.copy(),
        "historial_estado": historial_estado,
        "mejor_linea": mejor_linea_total,
        "estados_visitados": estados_visitados,
        "profundidad_maxima": profundidad_maxima
    })

#Resultados
    print(f"\n Estados analizados por la IA: {estados_visitados}")
    print(f" Profundidad maxima alcanzada: {profundidad_maxima}")


    if mejor_linea_total:
        print("\n Mejor linea estimada por la IA | jugador, posicion 1-9):")
# Convierte pos de 0-index a 1-9
        print([ (j, p+1) for (j,p) in mejor_linea_total ])

def historial():
    if not historial_partidas:
        print("No hay partidas jugadas para ser guardadas")
        return

    for idx, partida in enumerate(historial_partidas, 1):
        print(f"\n=== PARTIDA {idx} ===")
        print("Tablero final:")
        imprimir_tablero(partida["tablero_final"])

        print(f" Estados analizados: {partida.get('estados_visitados', 'N/A')}")
        print(f" Profundidad maxima: {partida.get('profundidad_maxima', 'N/A')}")


        if partida.get("mejor_linea"):
            print("\n Mejor linea (jugador, posicion 1-9):")
            print([ (j, p+1) for (j,p) in partida["mejor_linea"] ])

        print("\n Estados analizados por la IA:")
        for est_idx, estado in enumerate(partida["historial_estado"], 1):
            print(f"\n Estado {est_idx} | Profundidad: {estado['profundidad']} | Puntaje: {estado['puntaje']}")
            imprimir_tablero(estado["tablero"])


In [24]:
jugar()

=== TIC TAC TOE ===
Eres X, la computadora es O
┌───┬───┬───┐
│   │   │   │
├───┼───┼───┤
│   │   │   │
├───┼───┼───┤
│   │   │   │
└───┴───┴───┘
Elige tu movimiento 1-9: 1
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│   │   │   │
├───┼───┼───┤
│   │   │   │
└───┴───┴───┘
Turno de Computadora
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│   │ O │   │
├───┼───┼───┤
│   │   │   │
└───┴───┴───┘
Elige tu movimiento 1-9: 7
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│   │ O │   │
├───┼───┼───┤
│ X │   │   │
└───┴───┴───┘
Turno de Computadora
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│ O │ O │   │
├───┼───┼───┤
│ X │   │   │
└───┴───┴───┘
Elige tu movimiento 1-9: 6
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │   │   │
└───┴───┴───┘
Turno de Computadora
┌───┬───┬───┐
│ X │ O │   │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │   │   │
└───┴───┴───┘
Elige tu movimiento 1-9: 8
┌───┬───┬───┐
│ X │ O │   │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ X │   │
└───┴───┴──

In [25]:
historial()

Se han truncado las últimas 5000 líneas del flujo de salida.
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ O │ O │
└───┴───┴───┘

 Estado 60140 | Profundidad: 3 | Puntaje: None
┌───┬───┬───┐
│ X │   │ X │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │   │ O │
└───┴───┴───┘

 Estado 60141 | Profundidad: 4 | Puntaje: None
┌───┬───┬───┐
│ X │ O │ X │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │   │ O │
└───┴───┴───┘

 Estado 60142 | Profundidad: 5 | Puntaje: 0
┌───┬───┬───┐
│ X │ O │ X │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ X │ O │
└───┴───┴───┘

 Estado 60143 | Profundidad: 4 | Puntaje: None
┌───┬───┬───┐
│ X │   │ X │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ O │ O │
└───┴───┴───┘

 Estado 60144 | Profundidad: 5 | Puntaje: -5
┌───┬───┬───┐
│ X │ X │ X │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ O │ O │
└───┴───┴───┘

 Estado 60145 | Profundidad: 3 | Puntaje: None
┌───┬───┬───┐
│ X │   │   │
├───┼───┼───┤
│ O │ O │ X │
├───┼───┼───┤
│ X │ X │ O │
└───┴───┴───